<a href="https://colab.research.google.com/github/Rodrimansidub14/COMPUTER_VISION_L8/blob/main/TASK3_L8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CC3182 – Visión por Computadora | Laboratorio 8

**Dataset:** SKU110K — Dense Object Detection in Retail  
**Entorno:** Google Colab / Kaggle Notebooks  

---

### Estructura del notebook
1. [Configuración del entorno](#setup)
2. [Descarga y preparación del dataset](#dataset)
3. [Creación del subconjunto (train/val/test)](#subset)
4. [Verificación y visualización de anotaciones](#visualization)
5. [Entrenamiento — Modelo A](#modelA)
6. [Entrenamiento — Modelo B](#modelB)
7. [Evaluación comparativa](#evaluation)
8. [Visualización de detecciones](#detections)

---
## 1. Configuración del entorno <a id='setup'></a>

In [6]:
# Instalar dependencias necesarias
!pip install -q fiftyone ultralytics pycocotools opencv-python-headless matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.4 MB/s eta 0:0

In [7]:
import os
import json
import shutil
import random
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

SEED = 42
random.seed(SEED)
np.random.seed(SEED)



---
## 2. Descarga y preparación del dataset <a id='dataset'></a>

### Fuente
**SKU110K** es un dataset de detección densa de objetos en entornos retail. Contiene imágenes de anaqueles de supermercado con productos altamente apilados y solapados.

- Paper original: *"Precise Detection in Densely Packed Scenes"* (Goldman et al., CVPR 2019)
- Repositorio oficial: https://github.com/eg4000/SKU110K_CVPR19
- Descarga vía Kaggle: `trungdinh2212/sku110k`

### Proceso de descarga
Se utiliza la API de Kaggle para descargar el dataset directamente en el entorno de ejecución.  
Requiere tener `kaggle.json` con credenciales válidas montado en `/root/.kaggle/`.

In [16]:
import os, json
from pathlib import Path
from google.colab import userdata

# Leer secret y parsear JSON
creds = json.loads(userdata.get('KAGGLE_JSON'))

# Método 1: variables de entorno (más confiable en Colab)
# Las claves en un archivo kaggle.json suelen ser 'username' y 'key'.
# Asegúrate de que tu secreto KAGGLE_JSON solo contenga el JSON válido,
# sin texto adicional antes o después.
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY']      = creds['key']

# Método 2: kaggle.json como respaldo
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps(creds))
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print(f"Auth OK  →  username: {creds['username']}")
!kaggle datasets list --search sku110k

Auth OK  →  username: rodrigomansilla
ref                                                      title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
thedatasith/sku110k-annotations                          SKU110K Dataset                                     14123122669  2022-06-09 14:02:32.163000           4410         30                1  
rhtsingh/130k-images-512x512-universal-image-embeddings  130k Images (512x512) - Universal Image Embeddings  13900377408  2022-07-31 09:11:28.010000           2639         82        0.8235294  
rhtsingh/google-universal-image-embeddings-128x128       130k Images (128x128) - Universal Image Embeddings   1464524459  2022-07-29 09:22:40.090000           2692         32        0.82

In [18]:
DATA_ROOT = Path('data/SKU110K')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Descargar dataset (~13 GB completo; se tomará un subconjunto)
!kaggle datasets download -d thedatasith/sku110k-annotations -p {DATA_ROOT} --unzip

# Listar contenido descargado
for p in sorted(DATA_ROOT.rglob('*')):
    if p.is_dir():
        print(f'[DIR]  {p}')
    elif p.suffix in ('.csv', '.json', '.txt'):
        print(f'[FILE] {p}')

Dataset URL: https://www.kaggle.com/datasets/thedatasith/sku110k-annotations
License(s): Attribution-NonCommercial-ShareAlike 3.0 IGO (CC BY-NC-SA 3.0 IGO)
  9% 1.21G/13.2G [01:09<11:24, 18.7MB/s]
User cancelled operation
Exception ignored in atexit callback: <bound method TMonitor.exit of <TMonitor(Thread-1, stopped daemon 134524369741376)>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/_monitor.py", line 44, in exit
    self.join()
  File "/usr/lib/python3.12/threading.py", line 1149, in join
    self._wait_for_tstate_lock()
  File "/usr/lib/python3.12/threading.py", line 1169, in _wait_for_tstate_lock
    if lock.acquire(block, timeout):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


### Formato de anotaciones SKU110K

Las anotaciones originales vienen en archivos CSV con el esquema:
```
image_name, x1, y1, x2, y2, class, image_width, image_height
```
donde `(x1, y1)` es la esquina superior-izquierda y `(x2, y2)` la inferior-derecha del bounding box.  
Hay una única clase: **object** que representa el producto de anaquel.



In [ ]:
import pandas as pd

# Cargar CSV de anotaciones (ajustar ruta según estructura descargada)
ANNOT_TRAIN = DATA_ROOT / 'annotations' / 'annotations_train.csv'
ANNOT_VAL   = DATA_ROOT / 'annotations' / 'annotations_val.csv'
ANNOT_TEST  = DATA_ROOT / 'annotations' / 'annotations_test.csv'

COL_NAMES = ['image_name', 'x1', 'y1', 'x2', 'y2', 'class', 'img_w', 'img_h']

df_train = pd.read_csv(ANNOT_TRAIN, names=COL_NAMES)
df_val   = pd.read_csv(ANNOT_VAL,   names=COL_NAMES)
df_test  = pd.read_csv(ANNOT_TEST,  names=COL_NAMES)

print(f'Anotaciones originales  →  train: {df_train["image_name"].nunique()} imgs '
      f'| val: {df_val["image_name"].nunique()} imgs '
      f'| test: {df_test["image_name"].nunique()} imgs')

df_train.head(3)

---
## 3. Creación del subconjunto (train / val / test) <a id='subset'></a>

Se selecciona un subconjunto mínimo según los requisitos del laboratorio:
- **Train:** 500 imágenes
- **Val:** 100 imágenes
- **Test:** 100 imágenes

Las imágenes se muestrean aleatoriamente con semilla fija para reproducibilidad.  
Las anotaciones se convierten a formato YOLO y se guardan en la estructura de directorios estándar:
```
dataset/
  images/
    train/   val/   test/
  labels/
    train/   val/   test/
```

In [ ]:
# Tamaños del subconjunto
N_TRAIN = 500
N_VAL   = 100
N_TEST  = 100

DATASET_DIR = Path('dataset')
for split in ('train', 'val', 'test'):
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Estructura de directorios creada:')
for p in sorted(DATASET_DIR.rglob('*')):
    print(' ', p)

In [ ]:
def sample_images(df: pd.DataFrame, n: int, seed: int = SEED) -> list:
    """Muestrea n nombres de imagen únicos del dataframe."""
    unique_imgs = df['image_name'].unique().tolist()
    if n > len(unique_imgs):
        print(f'  Aviso: se solicitaron {n} imágenes pero solo hay {len(unique_imgs)}. Usando todas.')
        return unique_imgs
    random.seed(seed)
    return random.sample(unique_imgs, n)

train_imgs = sample_images(df_train, N_TRAIN)
val_imgs   = sample_images(df_val,   N_VAL)
test_imgs  = sample_images(df_test,  N_TEST)

print(f'Subconjunto seleccionado  →  train: {len(train_imgs)} | val: {len(val_imgs)} | test: {len(test_imgs)}')

In [ ]:
def bbox_to_yolo(row: pd.Series) -> str:
    """Convierte una fila CSV a línea YOLO normalizada."""
    x_c = ((row.x1 + row.x2) / 2) / row.img_w
    y_c = ((row.y1 + row.y2) / 2) / row.img_h
    w   = (row.x2 - row.x1) / row.img_w
    h   = (row.y2 - row.y1) / row.img_h
    # clase 0 = 'object' (única clase en SKU110K)
    return f'0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}'


def build_split(
    df: pd.DataFrame,
    img_names: list,
    split: str,
    src_img_dir: Path,
) -> dict:
    """Copia imágenes y escribe labels YOLO para un split dado."""
    stats = {'imgs_ok': 0, 'imgs_missing': 0, 'total_boxes': 0}
    df_split = df[df['image_name'].isin(img_names)]

    for img_name in img_names:
        src = src_img_dir / img_name
        dst_img = DATASET_DIR / 'images' / split / img_name
        dst_lbl = DATASET_DIR / 'labels' / split / (Path(img_name).stem + '.txt')

        # Copiar imagen
        if src.exists():
            shutil.copy2(src, dst_img)
            stats['imgs_ok'] += 1
        else:
            stats['imgs_missing'] += 1

        # Escribir anotaciones YOLO
        rows = df_split[df_split['image_name'] == img_name]
        lines = [bbox_to_yolo(r) for _, r in rows.iterrows()]
        dst_lbl.write_text('\n'.join(lines))
        stats['total_boxes'] += len(lines)

    return stats


# Directorio de imágenes originales (ajustar según estructura descargada)
SRC_IMG_TRAIN = DATA_ROOT / 'images' / 'train'
SRC_IMG_VAL   = DATA_ROOT / 'images' / 'val'
SRC_IMG_TEST  = DATA_ROOT / 'images' / 'test'

print('Construyendo split de entrenamiento...')
s_train = build_split(df_train, train_imgs, 'train', SRC_IMG_TRAIN)
print('Construyendo split de validación...')
s_val   = build_split(df_val,   val_imgs,   'val',   SRC_IMG_VAL)
print('Construyendo split de prueba...')
s_test  = build_split(df_test,  test_imgs,  'test',  SRC_IMG_TEST)

for split, s in [('train', s_train), ('val', s_val), ('test', s_test)]:
    print(f'  [{split}] imgs copiadas: {s["imgs_ok"]} | faltantes: {s["imgs_missing"]} | boxes: {s["total_boxes"]}')

In [ ]:
# Generar dataset.yaml para YOLO
yaml_content = f"""path: {DATASET_DIR.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names: ['object']
"""

(DATASET_DIR / 'dataset.yaml').write_text(yaml_content)
print('dataset.yaml generado:')
print(yaml_content)

In [ ]:
# Distribución de bounding boxes por imagen en cada split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (split, df_s, names) in zip(axes, [
    ('train', df_train, train_imgs),
    ('val',   df_val,   val_imgs),
    ('test',  df_test,  test_imgs),
]):
    counts = df_s[df_s['image_name'].isin(names)].groupby('image_name').size()
    ax.hist(counts, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(f'{split}  (n={len(names)} imgs, {counts.sum()} boxes)')
    ax.set_xlabel('Bounding boxes por imagen')
    ax.set_ylabel('Frecuencia')
    ax.axvline(counts.mean(), color='red', linestyle='--', label=f'Media: {counts.mean():.1f}')
    ax.legend()

plt.suptitle('Distribución de anotaciones en el subconjunto SKU110K', fontsize=13)
plt.tight_layout()
plt.savefig('distribution_boxes.png', dpi=120)
plt.show()
print('Gráfica guardada en distribution_boxes.png')

---
## 4. Verificación y visualización de anotaciones <a id='visualization'></a>

Se visualizan al menos 5 imágenes de entrenamiento con sus bounding boxes para confirmar que la conversión de anotaciones es correcta antes de proceder con el entrenamiento.

In [ ]:
def draw_yolo_boxes(img_path: Path, label_path: Path, ax, max_boxes: int = 80):
    """Dibuja bounding boxes YOLO sobre la imagen en un eje de matplotlib."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)

    if label_path.exists():
        lines = label_path.read_text().strip().splitlines()
        for line in lines[:max_boxes]:
            _, xc, yc, bw, bh = map(float, line.split())
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=1, edgecolor='lime', facecolor='none'
            )
            ax.add_patch(rect)
        ax.set_title(f'{img_path.name}\n{len(lines)} boxes', fontsize=8)
    else:
        ax.set_title(f'{img_path.name}\n[sin label]', fontsize=8)

    ax.axis('off')


# Seleccionar 5 imágenes aleatorias del split de entrenamiento
sample_names = random.sample(train_imgs, 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, name in zip(axes, sample_names):
    img_p = DATASET_DIR / 'images' / 'train' / name
    lbl_p = DATASET_DIR / 'labels' / 'train' / (Path(name).stem + '.txt')
    draw_yolo_boxes(img_p, lbl_p, ax)

plt.suptitle('Muestra de imágenes con anotaciones (formato YOLO)', fontsize=12)
plt.tight_layout()
plt.savefig('sample_annotations.png', dpi=120)
plt.show()
print('Visualización guardada en sample_annotations.png')

---
## 5. Entrenamiento — Modelo A <a id='modelA'></a>

> **Pendiente:** Selección de arquitectura, carga de pesos pre-entrenados, fine-tuning con capas base congeladas, Early Stopping.

---
## 6. Entrenamiento — Modelo B <a id='modelB'></a>

> **Pendiente:** Segunda arquitectura para comparación.

---
## 7. Evaluación comparativa <a id='evaluation'></a>

> **Pendiente:** mAP@0.5, mAP@0.5:0.95, Precisión, Recall, FPS, tamaño del modelo.

---
## 8. Visualización de detecciones <a id='detections'></a>

> **Pendiente:** Al menos 3 imágenes de prueba con bounding boxes predichos y score de confianza.